# 01. 텐서 기초

텐서(`torch.Tensor`)는 PyTorch의 기본 자료구조로, GPU에서도 연산할 수 있는 다차원 배열입니다.

**학습 목표**
- 텐서를 만들고 shape, dtype, device를 확인한다
- 기본 연산, 브로드캐스팅, 인덱싱, reshape를 익힌다
- NumPy 배열과 상호 변환하고 장치(device) 간에 옮긴다

In [1]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

PyTorch 2.14.1, device: cpu


## 1. 텐서 만들기

In [2]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(a)
print("shape:", a.shape, "| dtype:", a.dtype, "| device:", a.device)

tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3]) | dtype: torch.int64 | device: cpu


In [3]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3, dtype=torch.float64))
print(torch.arange(0, 10, 2))
print(torch.linspace(0, 1, 5))
print(torch.rand(2, 2))        # [0, 1) 균등분포
print(torch.randn(2, 2))       # 표준정규분포
print(torch.zeros_like(a))     # 같은 shape/dtype

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]], dtype=torch.float64)
tensor([0, 2, 4, 6, 8])
tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
tensor([[0.8823, 0.9150],
        [0.3829, 0.9593]])
tensor([[ 0.2345,  0.2303],
        [-1.1229, -0.1863]])
tensor([[0, 0, 0],
        [0, 0, 0]])


## 2. 기본 연산

산술 연산은 원소별(element-wise)로 동작하고, 행렬곱은 `@` 또는 `torch.matmul`을 씁니다.

In [4]:
x = torch.tensor([[1., 2.], [3., 4.]])
y = torch.tensor([[5., 6.], [7., 8.]])

print("x + y =\n", x + y)
print("x * y =\n", x * y)        # 원소별 곱
print("x @ y =\n", x @ y)        # 행렬곱
print("x.T =\n", x.T)
print("sum:", x.sum().item(), "| mean:", x.mean().item(), "| max:", x.max().item())
print("열 방향 합(dim=0):", x.sum(dim=0))
print("행 방향 합(dim=1):", x.sum(dim=1))

x + y =
 tensor([[ 6.,  8.],
        [10., 12.]])
x * y =
 tensor([[ 5., 12.],
        [21., 32.]])
x @ y =
 tensor([[19., 22.],
        [43., 50.]])
x.T =
 tensor([[1., 3.],
        [2., 4.]])
sum: 10.0 | mean: 2.5 | max: 4.0
열 방향 합(dim=0): tensor([4., 6.])
행 방향 합(dim=1): tensor([3., 7.])


## 3. 브로드캐스팅

shape이 다른 텐서끼리 연산하면, 크기가 1인 차원이 자동으로 늘어나 맞춰집니다.

In [5]:
m = torch.ones(3, 4)
row = torch.arange(4.)          # shape (4,)
col = torch.arange(3.).view(3, 1)  # shape (3, 1)

print("m + row =\n", m + row)
print("m + col =\n", m + col)
print("col * row (외적) =\n", col * row)

m + row =
 tensor([[1., 2., 3., 4.],
        [1., 2., 3., 4.],
        [1., 2., 3., 4.]])
m + col =
 tensor([[1., 1., 1., 1.],
        [2., 2., 2., 2.],
        [3., 3., 3., 3.]])
col * row (외적) =
 tensor([[0., 0., 0., 0.],
        [0., 1., 2., 3.],
        [0., 2., 4., 6.]])


## 4. 인덱싱과 슬라이싱

In [6]:
t = torch.arange(12).view(3, 4)
print(t)
print("t[0]      :", t[0])
print("t[:, 1]   :", t[:, 1])
print("t[1:, 2:] :\n", t[1:, 2:])
print("t[t > 5]  :", t[t > 5])          # 불리언 마스크
print("t[[0, 2]] :\n", t[[0, 2]])       # 정수 배열 인덱싱

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
t[0]      : tensor([0, 1, 2, 3])
t[:, 1]   : tensor([1, 5, 9])
t[1:, 2:] :
 tensor([[ 6,  7],
        [10, 11]])
t[t > 5]  : tensor([ 6,  7,  8,  9, 10, 11])
t[[0, 2]] :
 tensor([[ 0,  1,  2,  3],
        [ 8,  9, 10, 11]])


## 5. shape 바꾸기

`view`는 메모리를 공유하는 새 view를 만들고(연속 메모리 필요), `reshape`는 필요하면 복사합니다.

In [7]:
t = torch.arange(24)
print(t.view(2, 3, 4).shape)
print(t.reshape(4, -1).shape)          # -1은 자동 계산
print(t.view(2, 12).unsqueeze(0).shape)  # 차원 추가
print(torch.zeros(1, 3, 1).squeeze().shape)  # 크기 1인 차원 제거

a, b = torch.ones(2, 3), torch.zeros(2, 3)
print("cat dim=0:", torch.cat([a, b], dim=0).shape)
print("stack    :", torch.stack([a, b]).shape)
print("permute  :", torch.rand(2, 3, 4).permute(2, 0, 1).shape)

torch.Size([2, 3, 4])
torch.Size([4, 6])
torch.Size([1, 2, 12])
torch.Size([3])
cat dim=0: torch.Size([4, 3])
stack    : torch.Size([2, 2, 3])
permute  : torch.Size([4, 2, 3])


## 6. NumPy 변환

CPU 텐서와 NumPy 배열은 `from_numpy` / `.numpy()`로 변환하며 **메모리를 공유**합니다.

In [8]:
import numpy as np

arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
arr[0] = 100.0
print("메모리 공유:", t)

t2 = torch.tensor(arr)   # 복사본
arr[1] = 200.0
print("복사본    :", t2)
print("tensor -> numpy:", torch.ones(3).numpy())

메모리 공유: tensor([100.,   2.,   3.], dtype=torch.float64)
복사본    : tensor([100.,   2.,   3.], dtype=torch.float64)
tensor -> numpy: [1. 1. 1.]


## 7. device 이동

연산에 쓰이는 텐서는 모두 같은 장치에 있어야 합니다.

In [9]:
t = torch.rand(2, 2).to(device)
print(t.device)
result = (t @ t).cpu()   # 결과를 CPU로 가져오기
print(result)

cpu
tensor([[0.8354, 0.8319],
        [0.5104, 0.6186]])


## 요약
- `torch.tensor`, `zeros`, `rand` 등으로 텐서를 만들고 `shape`, `dtype`, `device`로 속성을 확인한다
- 연산은 원소별, 행렬곱은 `@`, shape이 다르면 브로드캐스팅된다
- `view`/`reshape`/`permute`로 모양을 바꾸고, `cat`/`stack`으로 합친다
- NumPy와는 메모리를 공유하며 변환되고, `.to(device)`로 장치를 옮긴다